# Data Collection Pipeline

Este notebook organiza la ingesta de transacciones y articulos, calcula la popularidad de cada articulo y sintetiza reseñas simuladas usando probabilidades basadas en las ventas. Cada seccion documenta un paso especifico de la pipeline para que sea sencillo mantener y reutilizar el flujo.

## 0. Configuracion e imports

Centralizamos constantes de rutas, parametros y utilidades reutilizables para garantizar reproducibilidad y claridad.

In [58]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from numpy.random import Generator

DATA_ROOT = Path("C:/Users/SPARTAN PC/Documents/Concentracion/proyecto")
TRANSACTIONS_PATH = DATA_ROOT / "transactions_train" / "transactions_train.csv"
ARTICLES_PATH = DATA_ROOT / "articles.csv" / "articles.csv"

POPULARITY_WEIGHT = 200
NOISE_SCALE = 0.005
MAX_REVIEW_PROB = 0.03
RNG = np.random.default_rng(seed=42)

def load_dataset(path: Path) -> pd.DataFrame:
    """Carga un CSV validando que exista el archivo."""
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo esperado: {path}")
    return pd.read_csv(path)

def compute_article_counts(transactions: pd.DataFrame) -> pd.DataFrame:
    """Cuenta transacciones por article_id y ordena de mayor a menor."""
    return (
        transactions.groupby("article_id")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

def filter_article_counts(counts: pd.DataFrame, threshold: Optional[int] = None) -> pd.DataFrame:
    """Permite descartar articulos con un conteo mayor al threshold."""
    if threshold is None:
        return counts
    return counts[counts["count"] <= threshold]

def compute_popularity(counts: pd.DataFrame, total_rows: int) -> pd.DataFrame:
    """Normaliza la columna count para obtener un popularity_score."""
    popularity = counts.copy()
    popularity["popularity_score"] = popularity["count"] / total_rows
    return popularity

def add_review_probability(df: pd.DataFrame, rng: Generator) -> pd.DataFrame:
    """Calcula p_review y review_flag de forma reproducible."""
    enriched = df.copy()
    enriched["p_review"] = (
        enriched["popularity_score"] * POPULARITY_WEIGHT
        + rng.random(len(enriched)) * NOISE_SCALE
    )
    enriched["p_review"] = enriched["p_review"].clip(0, MAX_REVIEW_PROB)
    enriched["review_flag"] = rng.random(len(enriched)) < enriched["p_review"]
    return enriched

def describe_review_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Devuelve conteos y porcentajes de review_flag."""
    summary = (
        df["review_flag"]
        .value_counts(dropna=False)
        .rename_axis("review_flag")
        .reset_index(name="count")
    )
    summary["percentage"] = (summary["count"] / len(df) * 100).round(2)
    return summary

def reviews_by_article(df: pd.DataFrame) -> pd.DataFrame:
    """Matriz con reseñas True/False por artículo."""
    return (
        df.groupby("article_id")["review_flag"]
        .value_counts()
        .unstack(fill_value=0)
        .rename(columns={False: "False", True: "True"})
    )

def reduce_reviews(
    df: pd.DataFrame,
    frac: float = 0.1,
    seed: int = 123,
) -> pd.DataFrame:
    """Devuelve un muestreo estratificado por article_id y review_flag."""
    rng = np.random.default_rng(seed)
    keep_indices = []

    for article_id, article_group in df.groupby("article_id"):
        for flag_value, flag_group in article_group.groupby("review_flag"):
            size = max(1, int(len(flag_group) * frac))
            # Si el grupo es muy pequeño, evita pedir más filas de las que existen.
            size = min(size, len(flag_group))
            if size == len(flag_group):
                keep_indices.extend(flag_group.index.tolist())
            else:
                sampled = rng.choice(flag_group.index.to_numpy(), size=size, replace=False)
                keep_indices.extend(sampled.tolist())

    return df.loc[keep_indices].reset_index(drop=True)

## 1. Carga de fuentes crudas

Leemos transacciones historicas y el catalogo de articulos usando las rutas definidas. Guardamos los `DataFrame` en memoria para que el resto de la pipeline trabaje siempre con datos consistentes.

In [44]:
df = load_dataset(TRANSACTIONS_PATH)
df_articles = load_dataset(ARTICLES_PATH)

print(f"Transacciones: {df.shape}")
print(f"Articulos: {df_articles.shape}")

Transacciones: (31788324, 5)
Articulos: (105542, 25)


### Exploracion rapida de las fuentes

Inspeccionamos el esquema y algunas filas para validar que las cargas sean correctas.

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31788324 entries, 0 to 31788323
Data columns (total 5 columns):
 #   Column            Dtype  
---  ------            -----  
 0   t_dat             object 
 1   customer_id       object 
 2   article_id        int64  
 3   price             float64
 4   sales_channel_id  int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 1.2+ GB


In [46]:
df.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [47]:
df_articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


## 2. Popularidad por articulo

Calculamos cuantas transacciones tiene cada articulo y derivamos un `popularity_score` normalizado para alimentar los pasos siguientes.

In [48]:
article_counts = compute_article_counts(df)
article_counts.head()

,article_id,count
0,706016001,50287
1,706016002,35043
2,372860001,31718
3,610776002,30199
4,759871002,26329


In [49]:
max_count = int(article_counts["count"].max())
article_counts_filtered = filter_article_counts(article_counts, threshold=max_count)
total_reviews = int(article_counts_filtered["count"].sum())

total_sales = len(df)
print(f"Total de transacciones: {total_sales:,}")
print(f"Mayor n�mero de transacciones por art�culo: {max_count:,}")
print(f"Transacciones consideradas tras el filtrado: {total_reviews:,}")

Total de transacciones: 31,788,324
Mayor n�mero de transacciones por art�culo: 50,287
Transacciones consideradas tras el filtrado: 31,788,324


In [50]:
popular_articles = compute_popularity(article_counts_filtered, total_rows=len(df))
popular_articles.head()

,article_id,count,popularity_score
0,706016001,50287,0.001582
1,706016002,35043,0.001102
2,372860001,31718,0.000998
3,610776002,30199,0.000950
4,759871002,26329,0.000828


## 3. Enriquecimiento y generacion de reseñas

Unimos las metricas de popularidad con las transacciones originales y generamos probabilidades/reseñas simuladas de forma reproducible.

In [51]:
df_full = df.merge(
    popular_articles,
    on="article_id",
    how="left",
    validate="many_to_one",
)

df_full = add_review_probability(df_full, rng=RNG)
df_full.head()

,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2,633,0.000020,0.007852,False
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2,434,0.000014,0.004925,False
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2,42,0.000001,0.004557,False
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2,1836,0.000058,0.015038,False
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2,1792,0.000056,0.011745,False


In [52]:
describe_review_flags(df_full)

,review_flag,count,percentage
0,False,31383604,98.73
1,True,404720,1.27


## 4. Dataset final de reseñas

Nos quedamos unicamente con las filas donde `review_flag` es `True`, incorporamos los metadatos de articulos y ofrecemos un par de vistas utiles para analisis posteriores.

In [53]:
df_reviews = df_full[df_full["review_flag"]].copy()
df_reviews_articles = df_reviews.merge(
    df_articles,
    on="article_id",
    how="left",
    validate="many_to_one",
)

print(f"Reseñas simuladas: {len(df_reviews_articles):,}")
df_reviews_articles.head()

Reseñas simuladas: 404,720


,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag,product_code,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,2018-09-20,001127bffdda108579e6cb16080440e89bf1250a776c6e...,397068015,0.033881,1,2409,0.000076,0.019318,True,397068,...,Denim trousers,F,Menswear,3,Menswear,56,Denim Men,1016,Trousers Denim,5-pocket low-rise jeans in washed stretch deni...
1,2018-09-20,014838e4a1fadaaaa5bd2504b73861ef89db7a86b1d0b1...,679121001,0.016932,1,879,0.000028,0.010361,True,679121,...,Expressive Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Soft, non-wired lace bra with moulded, lightly..."
2,2018-09-20,01c8cb2f730cd7253bd3e290d98823894222b305139631...,537688014,0.050831,1,3593,0.000113,0.024243,True,537688,...,Knitwear,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1003,Knitwear,Long polo-neck jumper in a soft knit with long...
3,2018-09-20,01ef5256e6264e9d50b11a08e9cfb2f2291324fedb4140...,220094001,0.015237,2,3011,0.000095,0.022025,True,220094,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Strapless maxi dress in jersey with an elastic...
4,2018-09-20,022d28daa94930566e6139ae25faa966bebd5684cc486c...,538142004,0.025407,2,585,0.000018,0.006710,True,538142,...,Denim Trousers,D,Divided,2,Divided,57,Ladies Denim,1016,Trousers Denim,5-pocket ankle-length jeans in washed stretch ...


In [54]:
review_counts = reviews_by_article(df_full)
review_counts.head()

review_flag,False,True
article_id,,
108775015,10510,331
108775044,7038,212
108775051,215,0
110065001,1035,9
110065002,532,7


In [56]:
review_counts.value_counts()

False  True
1      0       4474
2      0       3178
3      0       2618
4      0       2329
5      0       1966
               ... 
20341  672        1
20075  644        1
19782  682        1
19775  640        1
48754  1533       1
Name: count, Length: 12864, dtype: int64

# 5. Reducimos el Dataset de reviews_articles a el 10% (aprox. 40,000 items)

In [59]:
df_reviews_articles_v_reducido = reduce_reviews(df_reviews_articles, frac=0.1, seed=42)
print(len(df_reviews_articles), "→", len(df_reviews_articles_v_reducido))

404720 → 59458


In [60]:
original = df_reviews_articles.groupby("article_id")["review_flag"].mean()
reducido = df_reviews_articles_v_reducido.groupby("article_id")["review_flag"].mean()

diff = (original - reducido).abs().describe()
print(diff)


count    37383.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: review_flag, dtype: float64


## Comprobamos que el nuevo dataSet sea optimo y que cuente con suficientes reviews por articulo

In [63]:
review_counts_2 = reviews_by_article(df_reviews_articles_v_reducido)
review_counts_2

review_flag,True
article_id,
108775015,33
108775044,21
110065001,1
110065002,1
110065011,1
...,...
944506001,1
945995002,1
946748003,1


In [64]:
counts_per_article = review_counts_2["True"]

# Distribución absoluta y porcentual
dist = (
    counts_per_article
    .value_counts()            # cuántos artículos tienen 1, 2, 3 … reseñas
    .sort_index()
    .rename_axis("reseñas_por_articulo")
    .to_frame(name="articulos")
)
dist["porcentaje"] = (dist["articulos"] / dist["articulos"].sum() * 100).round(2)
dist["porcentaje_acumulado"] = dist["porcentaje"].cumsum().round(2)

dist


,articulos,porcentaje,porcentaje_acumulado
reseñas_por_articulo,,,
1,33335,89.17,89.17
2,1320,3.53,92.70
3,701,1.88,94.58
4,442,1.18,95.76
5,280,0.75,96.51
...,...,...,...
78,1,0.00,100.02
90,1,0.00,100.02
97,1,0.00,100.02


In [65]:
# guardar en csv a df_reviews_articles_v_reducido
df_reviews_articles_v_reducido.to_csv("df_reviews_articles_v_reducido.csv", index=False)

## Apendice  Utilitario para Groq

Dejamos una celda opcional que consulta el endpoint de Groq solo si la variable de entorno `GROQ_API_KEY` esta configurada.

In [55]:
# import os

# from groq import Groq

# def explain_fast_language_models(prompt: str = "Explain the importance of fast language models") -> str | None:
#     api_key = os.environ.get("GROQ_API_KEY")
#     if not api_key:
#         print("Configura GROQ_API_KEY para habilitar esta celda.")
#         return None

#     client = Groq(api_key=api_key)
#     completion = client.chat.completions.create(
#         messages=[{"role": "user", "content": prompt}],
#         model="llama-3.3-70b-versatile",
#     )
#     return completion.choices[0].message.content

# response = explain_fast_language_models()
# if response:
#     print(response)